# Modelagem com PyTorch — MLP para Predição de Churn

Notebook dedicado à construção, treinamento e avaliação de uma **Rede Neural (MLP)** utilizando PyTorch para o problema de churn.

**Etapas:**
1. Preparação dos Dados (replicando pipeline do `eda.ipynb`)
2. Arquitetura da MLP (PyTorch)
3. Loop de Treinamento com Batching e Early Stopping
4. Comparação MLP vs Modelos Baseline (6 métricas)
5. Análise de Trade-off de Custo: Falsos Positivos vs Falsos Negativos
6. Registro de Experimentos no MLflow

## 1. Setup e Preparação dos Dados

In [3]:
import pandas as pd
import numpy as np

# PyTorch
import torch

# MLflow
# import mlflow
# import mlflow.pytorch
# import mlflow.sklearn

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")

Dispositivo: cpu


### 1.1 Carregamento e Pré-processamento

Replicamos o mesmo pipeline de dados do `eda.ipynb` para garantir comparabilidade.

In [6]:
# ── Carregar dados ────────────────────────────────────────────────
df_churn = pd.read_csv("../data/Telco_customer_churn.csv")
print(f"Shape original: {df_churn.shape}")

# ── Remover colunas iniciais ──────────────────────────────────────
df_churn = df_churn.drop(columns=['Count', 'CustomerID', 'Churn Reason'])

# ── Feature Engineering ───────────────────────────────────────────
# 1. Perfil de alto risco: Fibra óptica + Contrato mensal
df_churn['high_risk_profile'] = (
    (df_churn['Internet Service'] == 'Fiber optic') &
    (df_churn['Contract'] == 'Month-to-month')
).astype(int)

# 2. Idoso isolado: Senior + Sem parceiro + Sem dependentes
df_churn['isolated_senior'] = (
    (df_churn['Senior Citizen'] == 'Yes') &
    (df_churn['Partner'] == 'No') &
    (df_churn['Dependents'] == 'No')
).astype(int)

# 3. Contagem de serviços de internet contratados
servicos = ['Online Security', 'Online Backup', 'Device Protection',
            'Tech Support', 'Streaming TV', 'Streaming Movies']
df_churn['internet_services_count'] = sum(
    (df_churn[c] == 'Yes').astype(int) for c in servicos
)

# 4. Custo por mês de permanência
df_churn['cost_per_month'] = df_churn['Monthly Charges'] / (df_churn['Tenure Months'] + 1)

# ── Conversão de tipos ────────────────────────────────────────────
for col in df_churn.columns:
    try:
        df_churn[col] = pd.to_numeric(df_churn[col])
    except (ValueError, TypeError):
        pass

# Total Charges: converter para float e preencher nulos
df_churn['Total Charges'] = pd.to_numeric(df_churn['Total Charges'], errors='coerce')
df_churn['Total Charges'] = df_churn['Total Charges'].fillna(df_churn['Total Charges'].median())

# ── Remover colunas de leakage e geográficas ─────────────────────
colunas_vazar = ['Churn Score', 'CLTV', 'Churn Label']
colunas_geograficas = ['City', 'Country', 'Lat Long', 'Latitude', 'Longitude', 'Zip Code']
colunas_drop = colunas_vazar + colunas_geograficas
df_churn = df_churn.drop(columns=[c for c in colunas_drop if c in df_churn.columns])

# ── One-Hot Encoding ──────────────────────────────────────────────
cat_cols = df_churn.select_dtypes(include=['object', 'category']).columns.tolist()
data_encoded = pd.get_dummies(df_churn, columns=cat_cols, drop_first=False)
bool_cols = data_encoded.select_dtypes(include=bool).columns
data_encoded[bool_cols] = data_encoded[bool_cols].astype(int)

# ── Preparação X e y ──────────────────────────────────────────────
df_model = data_encoded.select_dtypes(include=[np.number]).copy()
X = df_model.drop(columns=['Churn Value'])
y = df_model['Churn Value']
X = X.fillna(X.median())

print(f"Features disponíveis: {X.shape[1]}")
print(f"Distribuição do target:\n{y.value_counts(normalize=True).round(4)}")

Shape original: (7043, 33)
Features disponíveis: 51
Distribuição do target:
Churn Value
0    0.7346
1    0.2654
Name: proportion, dtype: float64
